# Stage 06-D: End-to-End Fine-Tuning Fusion

**Motivation:** Stages 06a/06b use frozen HDF5 features. Here we unfreeze the **top N PhoBERT
transformer layers** and train them jointly with a **deeper stacked cross-attention head**,
giving full gradient flow from classification loss back into the text encoder.

## Architecture

```
Article Text (tokens)                          COOLANT image [B, max_pairs, 64]
     |
PhoBERT (vinai/phobert-base-v2)
  bottom (12-N) layers: FROZEN
  top N layers:         trainable (lr = phobert_lr)
  CLS [B, 768]
     |
  TextProjector: Linear[768→proj_dim] + LN + ReLU      ImageProjector: Linear[64→proj_dim] + LN + ReLU
  query [B, 1, proj_dim]                               K, V [B, P, proj_dim]
     |
     +---- Stacked Cross-Attention (L layers, Pre-LN) ----+
              Each layer: x = x + CrossAttn(LN(x), K=LN(img), V=img)
                          x = x + FFN(LN(x))    FFN: Linear[d→4d] GELU Linear[4d→d]
     |
  context [B, proj_dim]
     |
  cat(h_t, context) [B, proj_dim*2]
     |
  MLP: [proj_dim*2 -> 256 -> 128 -> 3]
     |
  (Real=0, Fake=1, NEI=2)
```

## Key differences vs 06a

| | 06a | 06d |
|---|---|---|
| Text encoder | frozen HDF5 | PhoBERT, top N layers trainable |
| Cross-attn layers | 1 | N (default 2) |
| MLP depth | 1 hidden | 2 hidden |
| Trainable params | ~174K | ~7M (unf=3) |
| LR strategy | uniform 3e-4 | differential 1e-5 / 3e-4 |

**Inputs:** raw text (tokenized live) + `stage05a_cache` image features  
**Outputs:** `training/checkpoints_stage06d/`, `training/stage06d_results/`


In [ ]:
import os, sys
from pathlib import Path

def _detect_platform():
    try:
        import google.colab; return 'colab', False
    except ImportError: pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'): return 'vastai', False
    if Path('/workspace').exists(): return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True

PLATFORM, _ = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive; drive.mount('/content/drive')

try: _nb_path = Path(__file__).resolve()
except NameError: _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))
_env_map = {'colab': '.env.colab', 'vastai': '.env.vastai', 'windows': '.env.windows', 'mac': '.env.mac'}
_env_file = PROJECT_ROOT / _env_map.get(PLATFORM, '.env')
if not _env_file.exists(): _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f'Platform: {PLATFORM}  DATA_ROOT: {DATA_ROOT}  exists={DATA_ROOT.exists()}')

In [ ]:
# ── Step 1: Configuration ─────────────────────────────────────────────────
import os; from pathlib import Path
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'pipeline' else Path.cwd()
try:
    from dotenv import load_dotenv; load_dotenv(PROJECT_ROOT / '.env', override=False)
except ImportError: pass
DATA_ROOT = Path(os.environ['DATA_ROOT']) if os.environ.get('DATA_ROOT') else PROJECT_ROOT

CONFIG = {
    'paths': {
        'project_root':       PROJECT_ROOT,
        'stage05a_cache_dir': DATA_ROOT / 'training' / 'stage05a_cache',
        'data_json_dir':      DATA_ROOT / 'data',
        'checkpoint_root':    DATA_ROOT / 'training' / 'checkpoints_stage06d',
        'results_dir':        DATA_ROOT / 'training' / 'stage06d_results',
        'mlflow_dir':         DATA_ROOT / 'mlruns',
    },
    'phobert': {
        'model_name':      'vinai/phobert-base-v2',
        'max_seq_len':     256,
        'unfreeze_last_n': 3,   # 0=fully frozen (same as 06a), 3=~3.5M extra params
    },
    'model': {
        'arch_tag':        'e2e-finetune_phobert-coolant64_3cls',
        'image_dim':       64,
        'proj_dim':        256,
        'num_heads':       8,
        'num_attn_layers': 2,   # stacked cross-attention depth
        'mlp_hidden':      128,
        'num_classes':     3,
        'dropout':         0.2,
        'max_pairs':       32,  # must match stage05a_cache
    },
    'training': {
        'batch_size':      16,   # smaller: PhoBERT runs in the loop
        'max_epochs':      25,
        'patience':        6,
        'phobert_lr':      1e-5, # unfrozen PhoBERT layers — 30x lower than head
        'head_lr':         3e-4, # fresh fusion head
        'weight_decay':    1e-4,
        'label_smoothing': 0.1,
        'grad_clip':       1.0,
        'warmup_pct':      0.1,
        'seed':            42,
    },
    'data': {
        'hf_dataset':      'tranthaihoa/vifactcheck',
        # 3-class label mapping: Supported=0(Real), Refuted=1(Fake), NEI=2
        'label_map':       {0: 0, 1: 1, 2: 2},
    },
    'mlflow': {'experiment_name': 'stage06d-e2e-finetune'},
    'safety': {'smoke_test': False, 'smoke_batches': 3, 'smoke_epochs': 2},
}

CONFIG['paths']['checkpoint_root'].mkdir(parents=True, exist_ok=True)
CONFIG['paths']['results_dir'].mkdir(parents=True, exist_ok=True)
print(f'unfreeze_last_n={CONFIG["phobert"]["unfreeze_last_n"]}  '
      f'num_attn_layers={CONFIG["model"]["num_attn_layers"]}  '
      f'proj_dim={CONFIG["model"]["proj_dim"]}  '
      f'lr: phobert={CONFIG["training"]["phobert_lr"]} / head={CONFIG["training"]["head_lr"]}')

In [ ]:
# ── Step 2: Dependencies, Device, Seed ────────────────────────────────────
import importlib, sys, random, json, gc
from datetime import datetime

_required = [('torch','torch'),('h5py','h5py'),('numpy','numpy'),('pandas','pandas'),
             ('matplotlib','matplotlib'),('seaborn','seaborn'),('tqdm','tqdm'),
             ('sklearn','scikit-learn'),('transformers','transformers'),('datasets','datasets')]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing: raise RuntimeError(f'Missing: {_missing}')

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import h5py, matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModel

if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

def select_device():
    from src.utils.device import get_device; return get_device()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

device = select_device()
seed_everything(CONFIG['training']['seed'])
print(f'Device: {device}  PyTorch: {torch.__version__}')

In [ ]:
# ── Step 3: Load PhoBERT Tokenizer ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CONFIG['phobert']['model_name'], use_fast=True)
print(f'Tokenizer: {CONFIG["phobert"]["model_name"]}  vocab_size={tokenizer.vocab_size}')

In [ ]:
# ── Step 4: Load Text Data ─────────────────────────────────────────────────
def load_vifactcheck_split(split):
    """Returns list of {article_id, statement, evidence, label}. HF-first, local JSON fallback."""
    try:
        from datasets import load_dataset
        hf_split = {'dev': 'validation'}.get(split, split)
        ds = load_dataset(CONFIG['data']['hf_dataset'], split=hf_split)
        records = []
        for i, row in enumerate(ds):
            records.append({
                'article_id': str(row.get('id', row.get('article_id', i))),
                'statement':  str(row.get('claim', row.get('statement', ''))),
                'evidence':   str(row.get('evidence', '')),
                'label':      CONFIG['data']['label_map'][int(row['label'])],
            })
        print(f'  [{split}] HF: {len(records)} articles')
        return records
    except Exception as e:
        print(f'  [{split}] HF failed ({e}), trying local JSON...')

    for p in [CONFIG['paths']['data_json_dir'] / f'{split}.json',
              CONFIG['paths']['data_json_dir'] / f'vifactcheck_{split}.json',
              PROJECT_ROOT / 'data' / f'{split}.json']:
        if p.exists():
            with open(p) as f: data = json.load(f)
            records = [{'article_id': str(r.get('id', r.get('article_id', i))),
                        'statement':  str(r.get('claim', r.get('statement', ''))),
                        'evidence':   str(r.get('evidence', '')),
                        'label':      CONFIG['data']['label_map'][int(r['label'])]}
                       for i, r in enumerate(data)]
            print(f'  [{split}] Local ({p.name}): {len(records)} articles')
            return records
    raise FileNotFoundError(f'Cannot load {split} data.')

text_data = {s: load_vifactcheck_split(s) for s in ('train', 'dev', 'test')}
for split, recs in text_data.items():
    hist = np.bincount([r['label'] for r in recs], minlength=3).tolist()
    print(f'  [{split}] {len(recs)} articles  label_hist={hist}')

In [ ]:
# ── Step 5: Load Image Features from Stage 05a Cache ─────────────────────
cache_dir = CONFIG['paths']['stage05a_cache_dir']

def load_image_lookup(split):
    """Returns dict: article_id -> {image [max_pairs,64], pair_count, label}, id_source"""
    p = cache_dir / f'stage05a_{split}.h5'
    if not p.exists():
        raise FileNotFoundError(f'Stage 05a cache missing: {p}. Run 05a first.')
    with h5py.File(p, 'r') as f:
        img   = f['image_features'][:].astype(np.float32)
        cnts  = f['pair_counts'][:].astype(np.int32)
        lbls  = f['labels'][:].astype(np.int64)
        n     = int(f.attrs['n_articles'])
        mp    = int(f.attrs['max_pairs'])
        if 'article_ids' in f:
            aids = [s.decode() if isinstance(s, bytes) else str(s) for s in f['article_ids'][:]]
            src  = 'hdf5'
        else:
            aids = [str(i) for i in range(n)]
            src  = 'positional'
    if mp != CONFIG['model']['max_pairs']:
        raise ValueError(f'max_pairs mismatch: cache={mp} vs CONFIG={CONFIG["model"]["max_pairs"]}')
    lookup = {aid: {'image': img[i], 'pair_count': int(cnts[i]), 'label': int(lbls[i])}
              for i, aid in enumerate(aids)}
    print(f'  [{split}] n={n}  max_pairs={mp}  id_source={src}')
    return lookup, src

img_lookup = {}
id_sources = {}
for split in ('train', 'dev', 'test'):
    img_lookup[split], id_sources[split] = load_image_lookup(split)
print('\u2705 Image features loaded.')

In [ ]:
# ── Step 6: Align Text and Image ──────────────────────────────────────────
def align_split(text_recs, img_lkp, split, id_src):
    aligned, missing = [], 0
    for idx, rec in enumerate(text_recs):
        aid = rec['article_id']
        key = aid if aid in img_lkp else (str(idx) if id_src == 'positional' else None)
        if key is None or key not in img_lkp:
            missing += 1; continue
        ir = img_lkp[key]
        aligned.append({**rec, 'image': ir['image'], 'pair_count': ir['pair_count']})
    pct = 100 * len(aligned) / max(1, len(text_recs))
    print(f'  [{split}] matched={len(aligned)}/{len(text_recs)} ({pct:.1f}%)  missing={missing}')
    if pct < 50:
        raise RuntimeError(f'Only {pct:.1f}% matched. Check article_id format consistency.')
    return aligned

aligned = {s: align_split(text_data[s], img_lookup[s], s, id_sources[s])
           for s in ('train', 'dev', 'test')}
for split, recs in aligned.items():
    hist = np.bincount([r['label'] for r in recs], minlength=3).tolist()
    print(f'  [{split}] {len(recs)} articles  label_hist={hist}')

In [ ]:
# ── Step 7: Dataset + DataLoaders ─────────────────────────────────────────
class FinetuneFusionDataset(Dataset):
    def __init__(self, records, tokenizer, max_len):
        self.records = records; self.tokenizer = tokenizer; self.max_len = max_len

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        enc = self.tokenizer(
            rec['statement'], rec['evidence'],
            max_length=self.max_len, truncation='only_second',
            padding='max_length', return_tensors='pt',
        )
        return (
            enc['input_ids'].squeeze(0),
            enc['attention_mask'].squeeze(0),
            torch.from_numpy(rec['image'].astype(np.float32)),
            int(rec['pair_count']),
            int(rec['label']),
        )

def _collate(batch):
    ids, masks, imgs, cnts, lbls = zip(*batch)
    return (torch.stack(ids), torch.stack(masks), torch.stack(imgs),
            torch.tensor(cnts, dtype=torch.long), torch.tensor(lbls, dtype=torch.long))

max_len = CONFIG['phobert']['max_seq_len']
datasets_ = {s: FinetuneFusionDataset(aligned[s], tokenizer, max_len) for s in ('train','dev','test')}
bs = CONFIG['training']['batch_size']
loaders = {
    'train': DataLoader(datasets_['train'], batch_size=bs, shuffle=True,  num_workers=0, collate_fn=_collate),
    'dev':   DataLoader(datasets_['dev'],   batch_size=bs, shuffle=False, num_workers=0, collate_fn=_collate),
    'test':  DataLoader(datasets_['test'],  batch_size=bs, shuffle=False, num_workers=0, collate_fn=_collate),
}
if CONFIG['safety']['smoke_test']:
    from itertools import islice
    class _S:
        def __init__(self, l, n): self._l, self._n = l, n
        def __iter__(self): return islice(iter(self._l), self._n)
        def __len__(self): return min(self._n, len(self._l))
    loaders = {k: _S(v, CONFIG['safety']['smoke_batches']) for k, v in loaders.items()}

_ids, _masks, _img, _cnt, _lbl = next(iter(DataLoader(datasets_['train'], batch_size=2, collate_fn=_collate)))
print(f'Batch: ids={tuple(_ids.shape)}  img={tuple(_img.shape)}  labels={_lbl.tolist()}')

In [ ]:
# ── Step 8: Model ─────────────────────────────────────────────────────────
class StackedCrossAttention(nn.Module):
    """N Pre-LN cross-attention layers. Query updates; K/V fixed to projected image."""
    def __init__(self, proj_dim, num_heads, num_layers, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'ln_q':   nn.LayerNorm(proj_dim),
                'ln_kv':  nn.LayerNorm(proj_dim),
                'attn':   nn.MultiheadAttention(proj_dim, num_heads, dropout=dropout, batch_first=True),
                'ln_ffn': nn.LayerNorm(proj_dim),
                'ffn':    nn.Sequential(nn.Linear(proj_dim, proj_dim*4), nn.GELU(),
                                        nn.Dropout(dropout), nn.Linear(proj_dim*4, proj_dim)),
            }) for _ in range(num_layers)
        ])

    def forward(self, query, kv, key_pad_mask):
        # query: [B,1,d]  kv: [B,P,d]
        x, attn_w = query, None
        for layer in self.layers:
            q_ln = layer['ln_q'](x); kv_ln = layer['ln_kv'](kv)
            ao, attn_w = layer['attn'](q_ln, kv_ln, kv, key_padding_mask=key_pad_mask)
            x = x + ao
            x = x + layer['ffn'](layer['ln_ffn'](x))
        return x.squeeze(1), attn_w.squeeze(1)  # [B,d], [B,P]


class FinetuneFusionModel(nn.Module):
    def __init__(self, phobert_name, unfreeze_last_n,
                 image_dim=64, proj_dim=256, num_heads=8,
                 num_attn_layers=2, mlp_hidden=128, num_classes=3, dropout=0.2):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(phobert_name)

        # Freeze all, then selectively unfreeze
        for p in self.phobert.parameters(): p.requires_grad = False
        if unfreeze_last_n > 0:
            for layer in self.phobert.encoder.layer[-unfreeze_last_n:]:
                for p in layer.parameters(): p.requires_grad = True
            if hasattr(self.phobert, 'pooler'):
                for p in self.phobert.pooler.parameters(): p.requires_grad = True

        text_dim = self.phobert.config.hidden_size
        self.text_proj  = nn.Sequential(nn.Linear(text_dim, proj_dim), nn.LayerNorm(proj_dim), nn.ReLU())
        self.image_proj = nn.Sequential(nn.Linear(image_dim, proj_dim), nn.LayerNorm(proj_dim), nn.ReLU())
        self.cross_attn = StackedCrossAttention(proj_dim, num_heads, num_attn_layers, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim*2, mlp_hidden*2), nn.LayerNorm(mlp_hidden*2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden*2, mlp_hidden), nn.LayerNorm(mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, num_classes),
        )

    def forward(self, input_ids, attention_mask, image_feat, pair_counts):
        B, P, D = image_feat.shape
        cls = self.phobert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        h_t   = self.text_proj(cls)
        h_img = self.image_proj(image_feat.reshape(B*P, D)).reshape(B, P, -1)
        safe_cnt = pair_counts.clamp(min=1)
        idx = torch.arange(P, device=image_feat.device).unsqueeze(0)
        kpm = idx >= safe_cnt.unsqueeze(1).to(image_feat.device)
        ctx, attn_w = self.cross_attn(h_t.unsqueeze(1), h_img, kpm)
        return self.classifier(torch.cat([h_t, ctx], dim=-1)), attn_w


model = FinetuneFusionModel(
    phobert_name    = CONFIG['phobert']['model_name'],
    unfreeze_last_n = CONFIG['phobert']['unfreeze_last_n'],
    image_dim       = CONFIG['model']['image_dim'],
    proj_dim        = CONFIG['model']['proj_dim'],
    num_heads       = CONFIG['model']['num_heads'],
    num_attn_layers = CONFIG['model']['num_attn_layers'],
    mlp_hidden      = CONFIG['model']['mlp_hidden'],
    num_classes     = CONFIG['model']['num_classes'],
    dropout         = CONFIG['model']['dropout'],
).to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total: {total_p:,}  Trainable: {trainable_p:,} ({100*trainable_p/total_p:.1f}%)')
for name, mod in model.named_children():
    tp = sum(p.numel() for p in mod.parameters() if p.requires_grad)
    print(f'  {name}: trainable={tp:,}')

model.eval()
with torch.no_grad():
    _logits, _attn = model(_ids[:2].to(device), _masks[:2].to(device), _img[:2].to(device), _cnt[:2].to(device))
print(f'Sanity — logits:{tuple(_logits.shape)}  attn:{tuple(_attn.shape)}')
model.train()

In [ ]:
# ── Step 9: Differential LR Optimizer + Scheduler ─────────────────────────
nc = CONFIG['model']['num_classes']
train_labels = [r['label'] for r in aligned['train']]
counts_arr = np.bincount(train_labels, minlength=nc).astype(np.float64)
class_weights = torch.tensor(len(train_labels) / (nc * counts_arr), dtype=torch.float32).to(device)
print(f'Class weights: {[round(w,4) for w in class_weights.cpu().tolist()]}')

loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG['training']['label_smoothing'])

phobert_params = [p for p in model.phobert.parameters() if p.requires_grad]
head_params    = (list(model.text_proj.parameters()) + list(model.image_proj.parameters()) +
                  list(model.cross_attn.parameters()) + list(model.classifier.parameters()))

optimizer = torch.optim.AdamW(
    [{'params': phobert_params, 'lr': CONFIG['training']['phobert_lr']},
     {'params': head_params,    'lr': CONFIG['training']['head_lr']}],
    weight_decay=CONFIG['training']['weight_decay'],
)

total_steps = CONFIG['training']['max_epochs'] * len(loaders['train'])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[CONFIG['training']['phobert_lr'], CONFIG['training']['head_lr']],
    total_steps=total_steps,
    pct_start=CONFIG['training']['warmup_pct'],
    anneal_strategy='cos',
)
print(f'phobert_params={sum(p.numel() for p in phobert_params):,}  '
      f'head_params={sum(p.numel() for p in head_params):,}  '
      f'total_steps={total_steps}')

In [ ]:
# ── Step 10: Checkpoint Setup ─────────────────────────────────────────────
import mlflow

def _lr_s(lr): return f'{lr:.0e}'.replace('+0','').replace('-0','-').replace('e0','e')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = (f'{timestamp}_{CONFIG["model"]["arch_tag"]}'
            f'_unf{CONFIG["phobert"]["unfreeze_last_n"]}'
            f'_L{CONFIG["model"]["num_attn_layers"]}'
            f'_bs{CONFIG["training"]["batch_size"]}'
            f'_lr{_lr_s(CONFIG["training"]["head_lr"])}')
if CONFIG['safety']['smoke_test']: run_name = f'SMOKE_{run_name}'

run_dir = CONFIG['paths']['checkpoint_root'] / run_name
run_dir.mkdir(parents=True, exist_ok=True)
best_ckpt_path = run_dir / 'best_model.pth'
print(f'Run dir: {run_dir}')

def _jsonable(v):
    if isinstance(v, dict): return {k: _jsonable(x) for k,x in v.items()}
    if isinstance(v, Path): return str(v)
    if isinstance(v, (list, tuple)): return [_jsonable(x) for x in v]
    return v

def save_checkpoint(path, mdl, epoch, metrics, is_best):
    torch.save({'model_state_dict': mdl.state_dict(), 'config': _jsonable(CONFIG),
                'epoch': epoch, 'metrics': metrics, 'run_name': run_name,
                'phobert_name': CONFIG['phobert']['model_name'],
                'unfreeze_last_n': CONFIG['phobert']['unfreeze_last_n'],
                'num_attn_layers': CONFIG['model']['num_attn_layers'],
                'proj_dim': CONFIG['model']['proj_dim'],
                'num_classes': CONFIG['model']['num_classes'],
                'max_pairs': CONFIG['model']['max_pairs'],
                'is_best': is_best}, path)

mlflow_enabled = False
try:
    mlflow.set_tracking_uri(CONFIG['paths']['mlflow_dir'].as_uri())
    mlflow.set_experiment(CONFIG['mlflow']['experiment_name'])
    _run = mlflow.start_run(run_name=run_name)
    mlflow.log_params({'unfreeze_last_n': CONFIG['phobert']['unfreeze_last_n'],
                       'num_attn_layers': CONFIG['model']['num_attn_layers'],
                       'proj_dim': CONFIG['model']['proj_dim'],
                       'phobert_lr': CONFIG['training']['phobert_lr'],
                       'head_lr': CONFIG['training']['head_lr']})
    mlflow_enabled = True; print(f'MLflow run: {_run.info.run_id}')
except Exception as e:
    print(f'MLflow disabled ({type(e).__name__})')

In [ ]:
# ── Step 11: Training Loop ─────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, prefix, nc):
    acc = float(accuracy_score(y_true, y_pred))
    mf1 = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    pf1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    m = {f'{prefix}_accuracy': round(acc,4), f'{prefix}_macro_f1': round(mf1,4)}
    for i, cls in enumerate(['real','fake','nei']):
        m[f'{prefix}_f1_{cls}'] = round(float(pf1[i]) if i < len(pf1) else 0., 4)
    return m

def run_epoch(mdl, loader, loss_fn, optimizer, scheduler, device, train):
    mdl.train() if train else mdl.eval()
    total_loss, nb = 0., 0
    y_true, y_pred = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc='train' if train else 'eval ', leave=False):
            ids, masks, imgs, cnts, lbls = [x.to(device) for x in batch]
            logits, _ = mdl(ids, masks, imgs, cnts)
            loss = loss_fn(logits, lbls)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss: {loss.item()}')
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(mdl.parameters(), CONFIG['training']['grad_clip'])
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
            total_loss += loss.item(); nb += 1
            y_pred.extend(logits.argmax(-1).cpu().tolist())
            y_true.extend(lbls.cpu().tolist())
    return total_loss / max(1, nb), np.array(y_true), np.array(y_pred)


best_f1, best_ep, pat_ctr, history = -1., -1, 0, []
max_epochs = (CONFIG['safety']['smoke_epochs'] if CONFIG['safety']['smoke_test']
              else CONFIG['training']['max_epochs'])
print(f'Training: max_epochs={max_epochs}  patience={CONFIG["training"]["patience"]}  device={device}')

for epoch in range(1, max_epochs+1):
    tr_loss, yt, yp   = run_epoch(model, loaders['train'], loss_fn, optimizer, scheduler, device, True)
    vl_loss, yv, yp_v = run_epoch(model, loaders['dev'],   loss_fn, None,      None,      device, False)

    tr_m = compute_metrics(yt, yp,   'train', nc)
    vl_m = compute_metrics(yv, yp_v, 'val',   nc)
    row  = {'epoch': epoch, 'train_loss': round(tr_loss,4), 'val_loss': round(vl_loss,4), **tr_m, **vl_m}
    history.append(row)

    if mlflow_enabled:
        try: mlflow.log_metrics({'train_loss': tr_loss, 'val_loss': vl_loss, **tr_m, **vl_m}, step=epoch)
        except Exception: pass

    vf1 = vl_m['val_macro_f1']
    mark = ''
    if vf1 > best_f1:
        best_f1, best_ep, pat_ctr = vf1, epoch, 0
        save_checkpoint(best_ckpt_path, model, epoch, vl_m, True)
        mark = ' \u2190 best'
    else:
        pat_ctr += 1

    print(f'Epoch {epoch:02d}/{max_epochs} | tr={tr_loss:.4f} vl={vl_loss:.4f} | '
          f'val_f1={vf1:.4f} acc={vl_m["val_accuracy"]:.4f} '
          f'(pat {pat_ctr}/{CONFIG["training"]["patience"]}){mark}')

    if pat_ctr >= CONFIG['training']['patience']:
        print(f'Early stopping at epoch {epoch}'); break

print(f'\nBest epoch: {best_ep}  best val macro-F1: {best_f1:.4f}')

In [ ]:
# ── Step 12: Test Evaluation ───────────────────────────────────────────────
eval_model = FinetuneFusionModel(
    phobert_name    = CONFIG['phobert']['model_name'],
    unfreeze_last_n = CONFIG['phobert']['unfreeze_last_n'],
    image_dim       = CONFIG['model']['image_dim'],
    proj_dim        = CONFIG['model']['proj_dim'],
    num_heads       = CONFIG['model']['num_heads'],
    num_attn_layers = CONFIG['model']['num_attn_layers'],
    mlp_hidden      = CONFIG['model']['mlp_hidden'],
    num_classes     = CONFIG['model']['num_classes'],
    dropout         = CONFIG['model']['dropout'],
).to(device)
_c = torch.load(best_ckpt_path, map_location=device)
eval_model.load_state_dict(_c['model_state_dict'])
eval_model.eval()
print(f'Loaded best checkpoint (epoch {_c["epoch"]})')

te_loss, yt_te, yp_te = run_epoch(eval_model, loaders['test'], loss_fn, None, None, device, False)
test_m = compute_metrics(yt_te, yp_te, 'test', nc)
test_m['test_loss'] = round(te_loss, 4)
test_m['confusion_matrix'] = confusion_matrix(yt_te, yp_te, labels=[0,1,2]).tolist()

print('\nTest results:')
for k, v in test_m.items():
    if k != 'confusion_matrix': print(f'  {k}: {v}')
print(f'  confusion_matrix: {test_m["confusion_matrix"]}')

results = {
    'notebook':          '06d_finetune_fusion.ipynb',
    'proposal':          'D2 \u2014 End-to-End Fine-Tuning Fusion',
    'run_name':          run_name,
    'best_epoch':        best_ep,
    'best_val_macro_f1': best_f1,
    'val_metrics':       {k: v for k, v in history[best_ep-1].items() if k.startswith('val')},
    'test_metrics':      test_m,
    'training_history':  history,
    'arch':              _jsonable(CONFIG['model']),
    'phobert':           _jsonable(CONFIG['phobert']),
    'training':          _jsonable(CONFIG['training']),
}
results_path = CONFIG['paths']['results_dir'] / f'{run_name}_results.json'
with open(results_path, 'w') as f: json.dump(results, f, indent=2)
print(f'Results saved: {results_path}')

if mlflow_enabled:
    try: mlflow.log_metrics({k: v for k, v in test_m.items() if k != 'confusion_matrix'}); mlflow.end_run()
    except Exception: pass

In [ ]:
# ── Step 13: Training Curves + Confusion Matrix ────────────────────────────
if history:
    epochs = [r['epoch'] for r in history]
    color  = '#8B5CF6'
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (tr_k, vl_k, title) in zip(axes, [
        ('train_loss','val_loss','Loss'), ('train_accuracy','val_accuracy','Accuracy'),
        ('train_macro_f1','val_macro_f1','Macro F1')
    ]):
        ax.plot(epochs, [r[tr_k] for r in history], color=color, alpha=0.7, label='train')
        ax.plot(epochs, [r[vl_k] for r in history], color=color, linestyle='--', label='val')
        ax.axvline(best_ep, color='gray', linestyle=':', label=f'best ep{best_ep}')
        ax.set_title(f'06d Finetune \u2014 {title}'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    p = CONFIG['paths']['results_dir'] / f'{run_name}_curves.png'
    plt.savefig(p, dpi=120); plt.show(); plt.close()

if test_m.get('confusion_matrix'):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(np.array(test_m['confusion_matrix']), annot=True, fmt='d', cmap='Purples',
                xticklabels=['Real','Fake','NEI'], yticklabels=['Real','Fake','NEI'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title('Test Confusion Matrix \u2014 06d Finetune')
    plt.tight_layout()
    p = CONFIG['paths']['results_dir'] / f'{run_name}_confusion_matrix.png'
    plt.savefig(p, dpi=120); plt.show(); plt.close()

In [ ]:
# ── Step 14: Comparison vs Other Models ───────────────────────────────────
BASELINES = {tag: {'test_accuracy': None, 'test_macro_f1': None, 'test_f1_fake': None}
             for tag in ['05a MIL Attn', '05b Asym Gate', '05d Misinfo MIL', '06a Cross-Attn', '06b Decision']}

for tag, rd in [('05a MIL Attn','stage05a_results'),('05b Asym Gate','stage05b_results'),
                ('05d Misinfo MIL','stage05d_results'),('06a Cross-Attn','stage06a_results'),
                ('06b Decision','stage06b_results')]:
    d = DATA_ROOT / 'training' / rd
    if d.exists():
        cands = sorted(d.glob('*_results.json'), key=lambda p: p.stat().st_mtime, reverse=True)
        if cands:
            with open(cands[0]) as f: r = json.load(f)
            tm = r.get('test_metrics', {})
            BASELINES[tag] = {'test_accuracy': tm.get('test_accuracy'),
                              'test_macro_f1': tm.get('test_macro_f1'),
                              'test_f1_fake':  tm.get('test_f1_fake')}

print('='*62)
print(f'{"Model":<28} {"Acc":>8} {"F1":>8} {"F1-Fake":>10}')
print('-'*62)
for name, m in BASELINES.items():
    acc  = f'{m["test_accuracy"]:.4f}' if m['test_accuracy'] else '  N/A  '
    f1   = f'{m["test_macro_f1"]:.4f}' if m['test_macro_f1'] else '  N/A  '
    f1fk = f'{m["test_f1_fake"]:.4f}'  if m['test_f1_fake']  else '  N/A  '
    print(f'{name:<28} {acc:>8} {f1:>8} {f1fk:>10}')
print('-'*62)
print(f'{"06d Finetune (ours)":<28} '
      f'{test_m["test_accuracy"]:>8.4f} '
      f'{test_m["test_macro_f1"]:>8.4f} '
      f'{test_m.get("test_f1_fake",0):>10.4f}')
print('='*62)